### Extracting features of 2018 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2018 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2018)


In [257]:
import pandas as pd
import numpy as np
from tmdbv3api import TMDb
from tmdbv3api import Movie
import json
import requests
from io import StringIO


In [258]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2018"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2018 = tables[2]
df2_2018 = tables[3]
df3_2018 = tables[4]
df4_2018 = tables[5]

In [259]:
All_Scrabed= pd.concat([df1_2018, df2_2018, df3_2018, df4_2018], ignore_index=True)
All_Scrabed

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2]
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3]
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4]
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5]
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6]
...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[238]
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[142]
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[117]
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[207]


In [260]:
tmdb = TMDb()
tmdb.api_key = 'f8c7a842dd3a11ff444ffd2d20659eb0'

In [261]:
tmdb_movie = Movie()
def get_genre(x): 
    genres = []
    result = tmdb_movie.search(x)
    movie_id = result[0].id 
    response = requests.get('https://api.themoviedb.org/3/movie/{}?api_key={}'.format(movie_id,tmdb.api_key)) 
    data_json = response.json()
    if data_json['genres']: 
        genre_str = " " 
        for i in range(0,len(data_json['genres'])):
            genres.append(data_json['genres'][i]['name']) 
        return genre_str.join(genres)
    else:
        np.NaN


In [262]:
All_Scrabed['genres'] = All_Scrabed['Title'].map(lambda x: get_genre(str(x)))
All_Scrabed

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Drama Mystery
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller
...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[238],Romance Comedy
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[142],Comedy Mystery Crime
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[117],Drama Comedy
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[207],Drama History


In [263]:
Scrapped_2018 = All_Scrabed[['Title','Cast and crew','genres']]
Scrapped_2018

,Title,Cast and crew,genres
0,Insidious: The Last Key,Adam Robitel (director); Leigh Whannell (scree...,Horror Thriller
1,The Strange Ones,Christopher Radcliff (director/screenplay); La...,Drama Mystery
2,The Commuter,Jaume Collet-Serra (director); Byron Willinger...,Action Thriller Mystery
3,Proud Mary,"Babak Najafi (director); John S. Newman, Chris...",Thriller Action Crime
4,Acts of Violence,Brett Donowho (director); Nicolas Aaron Mezzan...,Action Crime Thriller
...,...,...,...
246,Second Act,"Peter Segal (director); Justin Zackham, Elaine...",Romance Comedy
247,Holmes & Watson,Etan Cohen (director/screenplay); Will Ferrell...,Comedy Mystery Crime
248,Vice,Adam McKay (director/screenplay); Christian Ba...,Drama Comedy
249,On the Basis of Sex,Mimi Leder (director); Daniel Stiepleman (scre...,Drama History


In [264]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x: 
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [265]:
Scrapped_2018['director_name'] = Scrapped_2018['Cast and crew'].map(lambda x: get_director(x))


In [266]:
Scrapped_2018

,Title,Cast and crew,genres,director_name
0,Insidious: The Last Key,Adam Robitel (director); Leigh Whannell (scree...,Horror Thriller,Adam Robitel
1,The Strange Ones,Christopher Radcliff (director/screenplay); La...,Drama Mystery,Christopher Radcliff (director/screenplay); La...
2,The Commuter,Jaume Collet-Serra (director); Byron Willinger...,Action Thriller Mystery,Jaume Collet-Serra
3,Proud Mary,"Babak Najafi (director); John S. Newman, Chris...",Thriller Action Crime,Babak Najafi
4,Acts of Violence,Brett Donowho (director); Nicolas Aaron Mezzan...,Action Crime Thriller,Brett Donowho
...,...,...,...,...
246,Second Act,"Peter Segal (director); Justin Zackham, Elaine...",Romance Comedy,Peter Segal
247,Holmes & Watson,Etan Cohen (director/screenplay); Will Ferrell...,Comedy Mystery Crime,Etan Cohen
248,Vice,Adam McKay (director/screenplay); Christian Ba...,Drama Comedy,Adam McKay
249,On the Basis of Sex,Mimi Leder (director); Daniel Stiepleman (scre...,Drama History,Mimi Leder


In [267]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [268]:
Scrapped_2018['actor_1_name'] = Scrapped_2018['Cast and crew'].map(lambda x: get_actor1(x))


In [269]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [270]:
Scrapped_2018['actor_2_name'] = Scrapped_2018['Cast and crew'].map(lambda x: get_actor2(x))

In [271]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [272]:
Scrapped_2018['actor_3_name'] = Scrapped_2018['Cast and crew'].map(lambda x: get_actor3(x))


In [273]:
Scrapped_2018

,Title,Cast and crew,genres,director_name,actor_1_name,actor_2_name,actor_3_name
0,Insidious: The Last Key,Adam Robitel (director); Leigh Whannell (scree...,Horror Thriller,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell
1,The Strange Ones,Christopher Radcliff (director/screenplay); La...,Drama Mystery,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus
2,The Commuter,Jaume Collet-Serra (director); Byron Willinger...,Action Thriller Mystery,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson
3,Proud Mary,"Babak Najafi (director); John S. Newman, Chris...",Thriller Action Crime,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown
4,Acts of Violence,Brett Donowho (director); Nicolas Aaron Mezzan...,Action Crime Thriller,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore
...,...,...,...,...,...,...,...
246,Second Act,"Peter Segal (director); Justin Zackham, Elaine...",Romance Comedy,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens
247,Holmes & Watson,Etan Cohen (director/screenplay); Will Ferrell...,Comedy Mystery Crime,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall
248,Vice,Adam McKay (director/screenplay); Christian Ba...,Drama Comedy,Adam McKay,Christian Bale,Amy Adams,Steve Carell
249,On the Basis of Sex,Mimi Leder (director); Daniel Stiepleman (scre...,Drama History,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux


In [274]:
Scrapped_2018 = Scrapped_2018.rename(columns={'Title':'movie_title'})


In [275]:
new_Scrapped_2018 = Scrapped_2018.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [276]:
new_Scrapped_2018

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,Insidious: The Last Key
1,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Drama Mystery,The Strange Ones
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,The Commuter
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,Proud Mary
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,Acts of Violence
...,...,...,...,...,...,...
246,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens,Romance Comedy,Second Act
247,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall,Comedy Mystery Crime,Holmes & Watson
248,Adam McKay,Christian Bale,Amy Adams,Steve Carell,Drama Comedy,Vice
249,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux,Drama History,On the Basis of Sex


In [277]:
new_Scrapped_2018['actor_2_name'] = new_Scrapped_2018['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2018['actor_3_name'] = new_Scrapped_2018['actor_3_name'].replace(np.nan, 'unknown')

In [278]:
new_Scrapped_2018['movie_title'] = new_Scrapped_2018['movie_title'].str.lower()

In [279]:
new_Scrapped_2018['comb'] = new_Scrapped_2018['actor_1_name'] + ' ' + new_Scrapped_2018['actor_2_name'] + ' '+ new_Scrapped_2018['actor_3_name'] + ' '+ new_Scrapped_2018['director_name'] +' ' + new_Scrapped_2018['genres']
new_Scrapped_2018

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,insidious: the last key,Lin Shaye Angus Sampson Leigh Whannell Adam Ro...
1,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Drama Mystery,the strange ones,Lauren Wolkstein (director); Alex Pettyfer Jam...
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,the commuter,Liam Neeson Vera Farmiga Patrick Wilson Jaume ...
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,proud mary,Taraji P. Henson Jahi Di'Allo Winston Billy Br...
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,acts of violence,Bruce Willis Cole Hauser Shawn Ashmore Brett D...
...,...,...,...,...,...,...,...
246,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens,Romance Comedy,second act,Jennifer Lopez Leah Remini Vanessa Hudgens Pet...
247,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall,Comedy Mystery Crime,holmes & watson,Will Ferrell John C. Reilly Rebecca Hall Etan ...
248,Adam McKay,Christian Bale,Amy Adams,Steve Carell,Drama Comedy,vice,Christian Bale Amy Adams Steve Carell Adam McK...
249,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux,Drama History,on the basis of sex,Felicity Jones Armie Hammer Justin Theroux Mim...


### Extracting features of 2019 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2019 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2019)

In [280]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2019"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2019 = tables[2]
df2_2019 = tables[3]
df3_2019 = tables[4]
df4_2019 = tables[5]

In [281]:
All_Scrabed_2 = pd.concat([df1_2019, df2_2019, df3_2019, df4_2019], ignore_index=True)
All_Scrabed_2

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,4,Escape Room,Columbia Pictures / Original Film,"Adam Robitel (director); Bragi F. Schut, Maria...",[2]
1,J A N U A R Y,4,Rust Creek,IFC Films / Lunacy Productions,Jen McGowan (director); Julie Lipson (screenpl...,[3]
2,J A N U A R Y,4,American Hangman,Hangman Justice Productions,Wilson Coneybeare (director/screenplay); Donal...,[4]
3,J A N U A R Y,11,A Dog's Way Home,Columbia Pictures,Charles Martin Smith (director); W. Bruce Came...,[5]
4,J A N U A R Y,11,The Upside,STX Entertainment,Neil Burger (director); Jon Hartmere (screenpl...,[6]
...,...,...,...,...,...,...
247,D E C E M B E R,25,Spies in Disguise,20th Century Fox Animation / Blue Sky Studios ...,"Nick Bruno, Troy Quane (directors); Brad Copel...",[133]
248,D E C E M B E R,25,Little Women,Columbia Pictures / Regency Enterprises,Greta Gerwig (director/screenplay); Saoirse Ro...,[228]
249,D E C E M B E R,25,1917,Universal Pictures / DreamWorks Pictures,Sam Mendes (director/screenplay); Krysty Wilso...,[229]
250,D E C E M B E R,25,Just Mercy,Warner Bros. Pictures / Participant,"Destin Daniel Cretton (director/screenplay), A...",[230]


In [282]:
All_Scrabed_2['genres'] = All_Scrabed_2['Title'].map(lambda x: get_genre(str(x)))

In [283]:
Scrapped_2019 = All_Scrabed_2[['Title','Cast and crew','genres']]
Scrapped_2019

,Title,Cast and crew,genres
0,Escape Room,"Adam Robitel (director); Bragi F. Schut, Maria...",Horror Thriller Mystery
1,Rust Creek,Jen McGowan (director); Julie Lipson (screenpl...,Thriller Drama Action Crime
2,American Hangman,Wilson Coneybeare (director/screenplay); Donal...,Thriller
3,A Dog's Way Home,Charles Martin Smith (director); W. Bruce Came...,Drama Adventure Family
4,The Upside,Neil Burger (director); Jon Hartmere (screenpl...,Comedy Drama
...,...,...,...
247,Spies in Disguise,"Nick Bruno, Troy Quane (directors); Brad Copel...",Animation Action Adventure Comedy Family
248,Little Women,Greta Gerwig (director/screenplay); Saoirse Ro...,Drama Romance
249,1917,Sam Mendes (director/screenplay); Krysty Wilso...,War History Drama
250,Just Mercy,"Destin Daniel Cretton (director/screenplay), A...",Drama Crime History


In [284]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [285]:
Scrapped_2019['director_name'] = Scrapped_2019['Cast and crew'].map(lambda x: get_director(str(x)))


In [286]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [287]:
Scrapped_2019['actor_1_name'] = Scrapped_2019['Cast and crew'].map(lambda x: get_actor1(str(x)))


In [288]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [289]:
Scrapped_2019['actor_2_name'] = Scrapped_2019['Cast and crew'].map(lambda x: get_actor2(str(x)))


In [290]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [291]:
Scrapped_2019['actor_3_name'] = Scrapped_2019['Cast and crew'].map(lambda x: get_actor3(str(x)))


In [292]:
Scrapped_2019 = Scrapped_2019.rename(columns={'Title':'movie_title'})

In [293]:
new_Scrapped_2019 = Scrapped_2019.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [294]:
new_Scrapped_2019['actor_2_name'] = new_Scrapped_2019['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2019['actor_3_name'] = new_Scrapped_2019['actor_3_name'].replace(np.nan, 'unknown')

In [295]:
new_Scrapped_2019['movie_title'] = new_Scrapped_2019['movie_title'].str.lower()

In [296]:
new_Scrapped_2019['comb'] = new_Scrapped_2019['actor_1_name'] + ' ' + new_Scrapped_2019['actor_2_name'] + ' '+ new_Scrapped_2019['actor_3_name'] + ' '+ new_Scrapped_2019['director_name'] +' ' + new_Scrapped_2019['genres']
new_Scrapped_2019

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Adam Robitel,Taylor Russell,Logan Miller,Deborah Ann Woll,Horror Thriller Mystery,escape room,Taylor Russell Logan Miller Deborah Ann Woll A...
1,Jen McGowan,Hermione Corfield,Jay Paulson,Sean O'Bryan,Thriller Drama Action Crime,rust creek,Hermione Corfield Jay Paulson Sean O'Bryan Jen...
2,Wilson Coneybeare,Donald Sutherland,Vincent Kartheiser,Oliver Dennis,Thriller,american hangman,Donald Sutherland Vincent Kartheiser Oliver De...
3,Charles Martin Smith,Bryce Dallas Howard,Edward James Olmos,Alexandra Shipp,Drama Adventure Family,a dog's way home,Bryce Dallas Howard Edward James Olmos Alexand...
4,Neil Burger,Bryan Cranston,Kevin Hart,Nicole Kidman,Comedy Drama,the upside,Bryan Cranston Kevin Hart Nicole Kidman Neil B...
...,...,...,...,...,...,...,...
247,"Nick Bruno, Troy Quane",Will Smith,Tom Holland,Rashida Jones,Animation Action Adventure Comedy Family,spies in disguise,Will Smith Tom Holland Rashida Jones Nick Brun...
248,Greta Gerwig,Saoirse Ronan,Emma Watson,Florence Pugh,Drama Romance,little women,Saoirse Ronan Emma Watson Florence Pugh Greta ...
249,Sam Mendes,George MacKay,Dean-Charles Chapman,Mark Strong,War History Drama,1917,George MacKay Dean-Charles Chapman Mark Strong...
250,Destin Daniel Cretton,Michael B. Jordan,Jamie Foxx,Brie Larson,Drama Crime History,just mercy,Michael B. Jordan Jamie Foxx Brie Larson Desti...


### Extracting features of 2020 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2020 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2020)

In [297]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2020"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2020= tables[2]
df2_2020 = tables[3]
df3_2020 = tables[4]
df4_2020 = tables[5]

In [298]:
All_Scrabed_3 = pd.concat([df1_2020, df2_2020, df3_2020, df4_2020], ignore_index=True)
All_Scrabed_3

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,3,The Grudge,Screen Gems / Stage 6 Films / Ghost House Pict...,Nicolas Pesce (director/screenplay); Andrea Ri...,[2]
1,J A N U A R Y,10,Underwater,20th Century Fox / Chernin Entertainment,"William Eubank (director); Brian Duffield, Ada...",[3]
2,J A N U A R Y,10,Like a Boss,Paramount Pictures / Artists First,"Miguel Arteta (director); Sam Pitman, Adam Col...",[4]
3,J A N U A R Y,10,Three Christs,IFC Films,Jon Avnet (director/screenplay); Eric Nazarian...,NaN
4,J A N U A R Y,10,Inherit the Viper,Lionsgate / Barry Films / Tycor International ...,Anthony Jerjen (director); Andrew Crabtree (sc...,[5]
...,...,...,...,...,...,...
274,D E C E M B E R,25,We Can Be Heroes,Netflix / Troublemaker Studios / Double R Prod...,Robert Rodriguez (director/screenplay); Priyan...,[247]
275,D E C E M B E R,25,News of the World,Universal Pictures / Playtone / Perfect World ...,Paul Greengrass (director/screenplay); Luke Da...,[248]
276,D E C E M B E R,25,One Night in Miami...,Amazon Studios,Regina King (director); Kemp Powers (screenpla...,[249]
277,D E C E M B E R,25,Promising Young Woman,Focus Features / FilmNation Entertainment,Emerald Fennell (director/screenplay); Carey M...,[250]


In [299]:
tmdb_movie = Movie()

def get_genre(x):
    genres = []

    result = tmdb_movie.search(x)

    if not result or not result.results:
        return np.nan

    movie_id = result.results[0].id

    response = requests.get(f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}")

    data_json = response.json()

    if data_json.get('genres'):
        for i in range(len(data_json['genres'])):
            genres.append(data_json['genres'][i]['name'])

        return " ".join(genres)

    return np.nan

In [300]:
All_Scrabed_3['genres'] = All_Scrabed_3['Title'].map(lambda x: get_genre(str(x)))

In [301]:
Scrapped_2020 = All_Scrabed_3[['Title','Cast and crew','genres']]
Scrapped_2020

,Title,Cast and crew,genres
0,The Grudge,Nicolas Pesce (director/screenplay); Andrea Ri...,Horror Mystery
1,Underwater,"William Eubank (director); Brian Duffield, Ada...",Horror Science Fiction Action Adventure
2,Like a Boss,"Miguel Arteta (director); Sam Pitman, Adam Col...",Comedy
3,Three Christs,Jon Avnet (director/screenplay); Eric Nazarian...,Drama
4,Inherit the Viper,Anthony Jerjen (director); Andrew Crabtree (sc...,Crime Thriller Drama
...,...,...,...
274,We Can Be Heroes,Robert Rodriguez (director/screenplay); Priyan...,Family Action Fantasy Comedy
275,News of the World,Paul Greengrass (director/screenplay); Luke Da...,Drama Western Adventure
276,One Night in Miami...,Regina King (director); Kemp Powers (screenpla...,Drama
277,Promising Young Woman,Emerald Fennell (director/screenplay); Carey M...,Thriller Crime Drama


In [302]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [303]:
Scrapped_2020['director_name'] = Scrapped_2020['Cast and crew'].map(lambda x: get_director(str(x)))

In [304]:
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

In [305]:
Scrapped_2020['actor_1_name'] = Scrapped_2020['Cast and crew'].map(lambda x: get_actor1(x))

In [306]:
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

In [307]:
Scrapped_2020['actor_2_name'] = Scrapped_2020['Cast and crew'].map(lambda x: get_actor2(x))


In [308]:
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

In [309]:
Scrapped_2020['actor_3_name'] = Scrapped_2020['Cast and crew'].map(lambda x: get_actor3(x))


In [310]:
Scrapped_2020 = Scrapped_2020.rename(columns={'Title':'movie_title'})

In [311]:
new_Scrapped_2020 = Scrapped_2020.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]


In [312]:
new_Scrapped_2020['actor_2_name'] = new_Scrapped_2020['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2020['actor_3_name'] = new_Scrapped_2020['actor_3_name'].replace(np.nan, 'unknown')

In [313]:
new_Scrapped_2020['movie_title'] = new_Scrapped_2020['movie_title'].str.lower()

In [314]:
new_Scrapped_2020['comb'] = new_Scrapped_2020['actor_1_name'] + ' ' + new_Scrapped_2020['actor_2_name'] + ' '+ new_Scrapped_2020['actor_3_name'] + ' '+ new_Scrapped_2020['director_name'] +' ' + new_Scrapped_2020['genres']
new_Scrapped_2020

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Nicolas Pesce,Andrea Riseborough,Demián Bichir,John Cho,Horror Mystery,the grudge,Andrea Riseborough Demián Bichir John Cho Nico...
1,William Eubank,Kristen Stewart,Vincent Cassel,Jessica Henwick,Horror Science Fiction Action Adventure,underwater,Kristen Stewart Vincent Cassel Jessica Henwick...
2,Miguel Arteta,Tiffany Haddish,Rose Byrne,Salma Hayek,Comedy,like a boss,Tiffany Haddish Rose Byrne Salma Hayek Miguel ...
3,Jon Avnet,Richard Gere,Peter Dinklage,Walton Goggins,Drama,three christs,Richard Gere Peter Dinklage Walton Goggins Jon...
4,Anthony Jerjen,Josh Hartnett,Margarita Levieva,Chandler Riggs,Crime Thriller Drama,inherit the viper,Josh Hartnett Margarita Levieva Chandler Riggs...
...,...,...,...,...,...,...,...
274,Robert Rodriguez,Priyanka Chopra Jonas,Pedro Pascal,YaYa Gosselin,Family Action Fantasy Comedy,we can be heroes,Priyanka Chopra Jonas Pedro Pascal YaYa Gossel...
275,Paul Greengrass,Tom Hanks,Helena Zengel,unknown,Drama Western Adventure,news of the world,Tom Hanks Helena Zengel unknown Paul Greengras...
276,Regina King,Kingsley Ben-Adir,Eli Goree,Aldis Hodge,Drama,one night in miami...,Kingsley Ben-Adir Eli Goree Aldis Hodge Regina...
277,Emerald Fennell,Carey Mulligan,Bo Burnham,Alison Brie,Thriller Crime Drama,promising young woman,Carey Mulligan Bo Burnham Alison Brie Emerald ...


### Extracting features of 2021 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2021 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2021)

In [315]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2021"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2021 = tables[2]
df2_2021 = tables[3]
df3_2021 = tables[4]
df4_2021 = tables[5]

In [316]:
All_Scrabed_4 = pd.concat([df1_2021, df2_2021, df3_2021, df4_2021], ignore_index=True)
All_Scrabed_4

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
0,J A N U A R Y,1.0,Shadow in the Cloud,Vertical Entertainment,Roseanne Liang (director/screenplay); Max Land...,[2]
1,J A N U A R Y,5.0,Hacksaw,Leone Films / Midnight Releasing,"Anthony Leone (director/screenplay); Amy Cay, ...",[3]
2,J A N U A R Y,12.0,Dr. Bird's Advice for Sad Poets,Relativity Media / Ketchup Entertainment,Yaniv Raz (director/screenplay); Lucas Jade Zu...,[4]
3,J A N U A R Y,13.0,The White Tiger,Netflix / ARRAY / Purple Pebble Pictures,Ramin Bahrani (director/screenplay); Adarsh Go...,NaN
4,J A N U A R Y,14.0,Locked Down,HBO Max / Warner Bros. Pictures,Doug Liman (director); Steven Knight (screenpl...,[5]
...,...,...,...,...,...,...
363,D E C E M B E R,25.0,The Tragedy of Macbeth,Apple TV+ / A24 / IAC Films,Joel Coen (director/screenplay); Denzel Washin...,[280]
364,D E C E M B E R,25.0,A Journal for Jordan,Columbia Pictures / Escape Artists / Bron Studios,Denzel Washington (director); Virgil Williams ...,[281]
365,D E C E M B E R,25.0,American Underdog,Lionsgate,"Erwin brothers (directors); Jon Erwin, David A...",[282]
366,D E C E M B E R,26.0,Memoria,Neon,Apichatpong Weerasethakul (director/acreenplay...,[283]


In [317]:
All_Scrabed_4['genres'] = All_Scrabed_4['Title'].map(lambda x: get_genre(str(x)))

In [318]:
Scrapped_2021 = All_Scrabed_4[['Title','Cast and crew','genres']]
Scrapped_2021

,Title,Cast and crew,genres
0,Shadow in the Cloud,Roseanne Liang (director/screenplay); Max Land...,Horror Action War
1,Hacksaw,"Anthony Leone (director/screenplay); Amy Cay, ...",Drama History War
2,Dr. Bird's Advice for Sad Poets,Yaniv Raz (director/screenplay); Lucas Jade Zu...,Comedy Drama
3,The White Tiger,Ramin Bahrani (director/screenplay); Adarsh Go...,Drama
4,Locked Down,Doug Liman (director); Steven Knight (screenpl...,Comedy Crime Romance
...,...,...,...
363,The Tragedy of Macbeth,Joel Coen (director/screenplay); Denzel Washin...,Drama
364,A Journal for Jordan,Denzel Washington (director); Virgil Williams ...,Drama Romance
365,American Underdog,"Erwin brothers (directors); Jon Erwin, David A...",Drama Family
366,Memoria,Apichatpong Weerasethakul (director/acreenplay...,Music Mystery


In [319]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [320]:
Scrapped_2021['director_name'] = Scrapped_2021['Cast and crew'].map(lambda x: get_director(str(x)))

In [321]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [322]:
Scrapped_2021['actor_1_name'] = Scrapped_2021['Cast and crew'].map(lambda x: get_actor1(x))

In [323]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [324]:
Scrapped_2021['actor_2_name'] = Scrapped_2021['Cast and crew'].map(lambda x: get_actor2(x))


In [325]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [326]:
Scrapped_2021['actor_3_name'] = Scrapped_2021['Cast and crew'].map(lambda x: get_actor3(x))


In [327]:
Scrapped_2021 = Scrapped_2021.rename(columns={'Title':'movie_title'})

In [328]:
new_Scrapped_2021 = Scrapped_2021.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [329]:
new_Scrapped_2021['actor_2_name'] = new_Scrapped_2021['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2021['actor_3_name'] = new_Scrapped_2021['actor_3_name'].replace(np.nan, 'unknown')

In [330]:
new_Scrapped_2021['movie_title'] = new_Scrapped_2021['movie_title'].str.lower()

In [331]:
new_Scrapped_2021['comb'] = new_Scrapped_2021['actor_1_name'] + ' ' + new_Scrapped_2021['actor_2_name'] + ' '+ new_Scrapped_2021['actor_3_name'] + ' '+ new_Scrapped_2021['director_name'] +' ' + new_Scrapped_2021['genres']
new_Scrapped_2021

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Roseanne Liang,Chloë Grace Moretz,Taylor John Smith,Beulah Koale,Horror Action War,shadow in the cloud,Chloë Grace Moretz Taylor John Smith Beulah Ko...
1,Anthony Leone,Amy Cay,Brian Patrick Butler,Michael C. Burgess,Drama History War,hacksaw,Amy Cay Brian Patrick Butler Michael C. Burges...
2,Yaniv Raz,Lucas Jade Zumann,Taylor Russell,Chase Stokes,Comedy Drama,dr. bird's advice for sad poets,Lucas Jade Zumann Taylor Russell Chase Stokes ...
3,Ramin Bahrani,Adarsh Gourav,Rajkummar Rao,Priyanka Chopra Jonas,Drama,the white tiger,Adarsh Gourav Rajkummar Rao Priyanka Chopra Jo...
4,Doug Liman,Anne Hathaway,Chiwetel Ejiofor,Stephen Merchant,Comedy Crime Romance,locked down,Anne Hathaway Chiwetel Ejiofor Stephen Merchan...
...,...,...,...,...,...,...,...
363,Joel Coen,Denzel Washington,Frances McDormand,Bertie Carvel,Drama,the tragedy of macbeth,Denzel Washington Frances McDormand Bertie Car...
364,Denzel Washington,Michael B. Jordan,Chanté Adams,Jalon Christian,Drama Romance,a journal for jordan,Michael B. Jordan Chanté Adams Jalon Christian...
365,Erwin brothers,Zachary Levi,Anna Paquin,Dennis Quaid,Drama Family,american underdog,Zachary Levi Anna Paquin Dennis Quaid Erwin br...
366,Apichatpong Weerasethakul (director/acreenplay...,Apichatpong Weerasethakul (director/acreenplay...,Elkin Díaz,Jeanne Balibar,Music Mystery,memoria,Apichatpong Weerasethakul (director/acreenplay...


### Extracting features of 2022 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2022 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2022)

In [332]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2022"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2022 = tables[2]
df2_2022 = tables[3]
df3_2022 = tables[4]
df4_2022 = tables[5]

In [333]:
All_Scrabed_5 = pd.concat([df1_2022, df2_2022, df3_2022, df4_2022], ignore_index=True)
All_Scrabed_5

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,7.0,The 355,Universal Pictures / Freckle Films / FilmNatio...,Simon Kinberg (director/screenplay); Theresa R...,[2]
1,J A N U A R Y,7.0,The Legend of La Llorona,Saban Films / Ageless Pictures,Patricia Harris Seeley (director); José Prende...,[3]
2,J A N U A R Y,7.0,The Commando,Saban Films / Premiere Entertainment,Asif Akbar (director); Koji Steven Sakai (scre...,[4]
3,J A N U A R Y,7.0,American Siege,Vertical Entertainment,Edward John Drake (director/screenplay); Timot...,[5]
4,J A N U A R Y,14.0,Scream,Paramount Pictures / Spyglass Media Group / Ra...,"Matt Bettinelli-Olpin, Tyler Gillett (director...",[6]
...,...,...,...,...,...,...
320,D E C E M B E R,30.0,"Alice, Darling",Lionsgate / Elevation Pictures,Mary Nighy (director); Alanna Francis (screenp...,[268]
321,D E C E M B E R,NaN,NaN,NaN,NaN,NaN
322,D E C E M B E R,NaN,NaN,NaN,NaN,NaN
323,D E C E M B E R,NaN,NaN,NaN,NaN,NaN


In [334]:
All_Scrabed_5['genres'] = All_Scrabed_5['Title'].map(lambda x: get_genre(str(x)))

In [335]:
Scrapped_2022 = All_Scrabed_5[['Title','Cast and crew','genres']]
Scrapped_2022

,Title,Cast and crew,genres
0,The 355,Simon Kinberg (director/screenplay); Theresa R...,Action Adventure Thriller
1,The Legend of La Llorona,Patricia Harris Seeley (director); José Prende...,Horror Thriller
2,The Commando,Asif Akbar (director); Koji Steven Sakai (scre...,Action Crime Thriller
3,American Siege,Edward John Drake (director/screenplay); Timot...,Action Adventure Thriller
4,Scream,"Matt Bettinelli-Olpin, Tyler Gillett (director...",Horror Mystery Crime
...,...,...,...
320,"Alice, Darling",Mary Nighy (director); Alanna Francis (screenp...,Thriller Drama Romance
321,NaN,NaN,Horror Thriller
322,NaN,NaN,Horror Thriller
323,NaN,NaN,Horror Thriller


In [336]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [337]:
Scrapped_2022['director_name'] = Scrapped_2022['Cast and crew'].map(lambda x: get_director(str(x)))

In [338]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [339]:
Scrapped_2022['actor_1_name'] = Scrapped_2022['Cast and crew'].map(lambda x: get_actor1(x))

In [340]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [341]:
Scrapped_2022['actor_2_name'] = Scrapped_2022['Cast and crew'].map(lambda x: get_actor2(x))

In [342]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [343]:
Scrapped_2022['actor_3_name'] = Scrapped_2022['Cast and crew'].map(lambda x: get_actor3(x))

In [344]:
Scrapped_2022 = Scrapped_2022.rename(columns={'Title':'movie_title'})

In [345]:
new_Scrapped_2022 = Scrapped_2022.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [346]:
new_Scrapped_2022['actor_2_name'] = new_Scrapped_2022['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2022['actor_3_name'] = new_Scrapped_2022['actor_3_name'].replace(np.nan, 'unknown')

In [347]:
new_Scrapped_2022['movie_title'] = new_Scrapped_2022['movie_title'].str.lower()

In [348]:
new_Scrapped_2022['comb'] = new_Scrapped_2022['actor_1_name'] + ' ' + new_Scrapped_2022['actor_2_name'] + ' '+ new_Scrapped_2022['actor_3_name'] + ' '+ new_Scrapped_2022['director_name'] +' ' + new_Scrapped_2022['genres']
new_Scrapped_2022

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Simon Kinberg,Jessica Chastain,Penélope Cruz,Fan Bingbing,Action Adventure Thriller,the 355,Jessica Chastain Penélope Cruz Fan Bingbing Si...
1,Patricia Harris Seeley,Autumn Reeser,Danny Trejo,Antonio Cupo,Horror Thriller,the legend of la llorona,Autumn Reeser Danny Trejo Antonio Cupo Patrici...
2,Asif Akbar,Mickey Rourke,Michael Jai White,unknown,Action Crime Thriller,the commando,Mickey Rourke Michael Jai White unknown Asif A...
3,Edward John Drake,Timothy V. Murphy,Bruce Willis,Rob Gough,Action Adventure Thriller,american siege,Timothy V. Murphy Bruce Willis Rob Gough Edwar...
4,"Matt Bettinelli-Olpin, Tyler Gillett",Melissa Barrera,Mason Gooding,Jenna Ortega,Horror Mystery Crime,scream,Melissa Barrera Mason Gooding Jenna Ortega Mat...
...,...,...,...,...,...,...,...
320,Mary Nighy,Anna Kendrick,Kaniehtiio Horn,Charlie Carrick,Thriller Drama Romance,"alice, darling",Anna Kendrick Kaniehtiio Horn Charlie Carrick ...
321,nan,NaN,unknown,unknown,Horror Thriller,NaN,NaN
322,nan,NaN,unknown,unknown,Horror Thriller,NaN,NaN
323,nan,NaN,unknown,unknown,Horror Thriller,NaN,NaN


### Extracting features of 2023 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2023 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2023)

In [349]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2023"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2023 = tables[2]
df2_2023 = tables[3]
df3_2023 = tables[4]
df4_2023 = tables[5]

In [350]:
All_Scrabed_6 = pd.concat([df1_2023, df2_2023, df3_2023, df4_2023], ignore_index=True)
All_Scrabed_6

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,6,M3GAN,Universal Pictures / Blumhouse Productions / A...,Gerard Johnstone (director); Akela Cooper (scr...,[3]
1,J A N U A R Y,6,The Old Way,Saban Films / Saturn Films,Brett Donowho (director); Carl W. Lucas (scree...,[4]
2,J A N U A R Y,10,Mystery Highway,TimeAxis Entertainment / Intellexual Entertain...,Clarke M. Smith (director/screenplay); David S...,[5]
3,J A N U A R Y,11,The Devil Conspiracy,Samuel Goldwyn Films,Nathan Frankowski (director); Ed Alan (screenp...,[6]
4,J A N U A R Y,13,Plane,Lionsgate / MadRiver Pictures / Di Bonaventura...,Jean-François Richet (director); Charles Cummi...,[7]
...,...,...,...,...,...,...
342,D E C E M B E R,22,Memory,Ketchup Entertainment / Mubi,Michel Franco (director/screenplay); Jessica C...,[330]
343,D E C E M B E R,25,The Color Purple,Warner Bros. Pictures / Amblin Entertainment /...,"Blitz Bazawule (director), Marcus Gardley (scr...",[331]
344,D E C E M B E R,25,The Boys in the Boat,Metro-Goldwyn-Mayer / Smokehouse Pictures,"George Clooney (director), Mark L. Smith (scre...",[332]
345,D E C E M B E R,25,Ferrari,Neon / STXfilms / Ketchup Entertainment,"Michael Mann (director), Troy Kennedy Martin (...",[333]


In [351]:
All_Scrabed_6['genres'] = All_Scrabed_6['Title'].map(lambda x: get_genre(str(x)))

In [352]:
Scrapped_2023 = All_Scrabed_6[['Title','Cast and crew','genres']]
Scrapped_2023

,Title,Cast and crew,genres
0,M3GAN,Gerard Johnstone (director); Akela Cooper (scr...,Science Fiction Horror
1,The Old Way,Brett Donowho (director); Carl W. Lucas (scree...,Action Drama Western
2,Mystery Highway,Clarke M. Smith (director/screenplay); David S...,Adventure
3,The Devil Conspiracy,Nathan Frankowski (director); Ed Alan (screenp...,Horror Fantasy Science Fiction Thriller
4,Plane,Jean-François Richet (director); Charles Cummi...,Horror Comedy
...,...,...,...
342,Memory,Michel Franco (director/screenplay); Jessica C...,Action Thriller Crime
343,The Color Purple,"Blitz Bazawule (director), Marcus Gardley (scr...",Drama History
344,The Boys in the Boat,"George Clooney (director), Mark L. Smith (scre...",Drama History
345,Ferrari,"Michael Mann (director), Troy Kennedy Martin (...",History Drama


In [353]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [354]:
Scrapped_2023['director_name'] = Scrapped_2023['Cast and crew'].map(lambda x: get_director(str(x)))

In [355]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [356]:
Scrapped_2023['actor_1_name'] = Scrapped_2023['Cast and crew'].map(lambda x: get_actor1(x))

In [357]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [358]:
Scrapped_2023['actor_2_name'] = Scrapped_2023['Cast and crew'].map(lambda x: get_actor2(x))

In [359]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [360]:
Scrapped_2023['actor_3_name'] = Scrapped_2023['Cast and crew'].map(lambda x: get_actor3(x))

In [361]:
Scrapped_2023 = Scrapped_2023.rename(columns={'Title':'movie_title'})

In [362]:
new_Scrapped_2023 = Scrapped_2023.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [363]:
new_Scrapped_2023['actor_2_name'] = new_Scrapped_2023['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2023['actor_3_name'] = new_Scrapped_2023['actor_3_name'].replace(np.nan, 'unknown')

In [364]:
new_Scrapped_2023['movie_title'] = new_Scrapped_2023['movie_title'].str.lower()

In [365]:
new_Scrapped_2023['comb'] = new_Scrapped_2023['actor_1_name'] + ' ' + new_Scrapped_2023['actor_2_name'] + ' '+ new_Scrapped_2023['actor_3_name'] + ' '+ new_Scrapped_2023['director_name'] +' ' + new_Scrapped_2023['genres']
new_Scrapped_2023

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Gerard Johnstone,Allison Williams,Violet McGraw,Amie Donald,Science Fiction Horror,m3gan,Allison Williams Violet McGraw Amie Donald Ger...
1,Brett Donowho,Nicolas Cage,Ryan Kiera Armstrong,unknown,Action Drama Western,the old way,Nicolas Cage Ryan Kiera Armstrong unknown Bret...
2,Clarke M. Smith,Rachel Elizabeth Ames,Christopher Cendana,Randy Davison,Adventure,mystery highway,Rachel Elizabeth Ames Christopher Cendana Rand...
3,Nathan Frankowski,Alice Orr-Ewing,Joe Doyle,Eveline Hall,Horror Fantasy Science Fiction Thriller,the devil conspiracy,Alice Orr-Ewing Joe Doyle Eveline Hall Nathan ...
4,Jean-François Richet,Gerard Butler,Mike Colter,Yoson An,Horror Comedy,plane,Gerard Butler Mike Colter Yoson An Jean-Franço...
...,...,...,...,...,...,...,...
342,Michel Franco,Jessica Chastain,Peter Sarsgaard,Merritt Wever,Action Thriller Crime,memory,Jessica Chastain Peter Sarsgaard Merritt Wever...
343,Blitz Bazawule,Fantasia Barrino,Taraji P. Henson,Danielle Brooks,Drama History,the color purple,Fantasia Barrino Taraji P. Henson Danielle Bro...
344,George Clooney,Callum Turner,Joel Edgerton,Peter Guinness,Drama History,the boys in the boat,Callum Turner Joel Edgerton Peter Guinness Geo...
345,Michael Mann,Adam Driver,Penelope Cruz,Shailene Woodley,History Drama,ferrari,Adam Driver Penelope Cruz Shailene Woodley Mic...


### Extracting features of 2024 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2024 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2024)

In [366]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2024"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2024 = tables[2]
df2_2024 = tables[3]
df3_2024 = tables[4]
df4_2024 = tables[5]

In [367]:
All_Scrabed_7 = pd.concat([df1_2024, df2_2024, df3_2024, df4_2024], ignore_index=True)
All_Scrabed_7

,Opening,Opening.1,Title,Company credits,Cast and crew,Ref.
0,J A N U A R Y,2.0,The Mummy Murders,Gravitas Ventures,Colin Bressler (director/screenplay); Will Don...,[3]
1,J A N U A R Y,3.0,Self Reliance,Neon / Hulu / MRC / Paramount Global Content D...,Jake Johnson (director/screenplay); Jake Johns...,[4]
2,J A N U A R Y,4.0,DarkGame,Gravitas Ventures,"Howard J. Ford (director); Gary Grant, Niall J...",[5]
3,J A N U A R Y,5.0,Night Swim,Universal Pictures / Blumhouse Productions / A...,Bryce McGuire (director/screenplay); Wyatt Rus...,[6]
4,J A N U A R Y,5.0,He Went That Way,Vertical Entertainment / Mister Smith Entertai...,Jeffrey Darling (director); Evan M. Wiener (sc...,[7]
...,...,...,...,...,...,...
490,D E C E M B E R,25.0,A Complete Unknown,Searchlight Pictures / Veritas Entertainment /...,James Mangold (director/screenplay); Jay Cocks...,[465]
491,D E C E M B E R,25.0,The Fire Inside,Metro-Goldwyn-Mayer,Rachel Morrison (director); Barry Jenkins (scr...,[466]
492,D E C E M B E R,25.0,Babygirl,A24 / 2AM,Halina Reijn (director/screenplay); Nicole Kid...,[467]
493,D E C E M B E R,25.0,Los Frikis,Wayward/Range / Lord Miller,"Michael Schwartz, Tyler Nilson (directors/scre...",[468]


In [368]:
All_Scrabed_7['genres'] = All_Scrabed_7['Title'].map(lambda x: get_genre(str(x)))

In [369]:
Scrapped_2024 = All_Scrabed_7[['Title','Cast and crew','genres']]
Scrapped_2024

,Title,Cast and crew,genres
0,The Mummy Murders,Colin Bressler (director/screenplay); Will Don...,Horror Crime
1,Self Reliance,Jake Johnson (director/screenplay); Jake Johns...,Comedy Thriller
2,DarkGame,"Howard J. Ford (director); Gary Grant, Niall J...",Horror Thriller
3,Night Swim,Bryce McGuire (director/screenplay); Wyatt Rus...,Horror Mystery
4,He Went That Way,Jeffrey Darling (director); Evan M. Wiener (sc...,Thriller Crime Drama
...,...,...,...
490,A Complete Unknown,James Mangold (director/screenplay); Jay Cocks...,Drama Music
491,The Fire Inside,Rachel Morrison (director); Barry Jenkins (scr...,Drama History
492,Babygirl,Halina Reijn (director/screenplay); Nicole Kid...,Romance Thriller
493,Los Frikis,"Michael Schwartz, Tyler Nilson (directors/scre...",Drama


In [370]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [371]:
Scrapped_2024['director_name'] = Scrapped_2024['Cast and crew'].map(lambda x: get_director(str(x)))

In [372]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [373]:
Scrapped_2024['actor_1_name'] = Scrapped_2024['Cast and crew'].map(lambda x: get_actor1(x))

In [374]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [375]:
Scrapped_2024['actor_2_name'] = Scrapped_2024['Cast and crew'].map(lambda x: get_actor2(x))

In [376]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [377]:
Scrapped_2024['actor_3_name'] = Scrapped_2024['Cast and crew'].map(lambda x: get_actor3(x))

In [378]:
Scrapped_2024 = Scrapped_2024.rename(columns={'Title':'movie_title'})

In [379]:
new_Scrapped_2024 = Scrapped_2024.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [380]:
new_Scrapped_2024['actor_2_name'] = new_Scrapped_2024['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2024['actor_3_name'] = new_Scrapped_2024['actor_3_name'].replace(np.nan, 'unknown')

In [381]:
new_Scrapped_2024['movie_title'] = new_Scrapped_2024['movie_title'].str.lower()

In [382]:
new_Scrapped_2024['comb'] = new_Scrapped_2024['actor_1_name'] + ' ' + new_Scrapped_2024['actor_2_name'] + ' '+ new_Scrapped_2024['actor_3_name'] + ' '+ new_Scrapped_2024['director_name'] +' ' + new_Scrapped_2024['genres']
new_Scrapped_2024

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Colin Bressler,Leila Annastasia Scott,Jason Scarbrough,Jeff Caperton,Horror Crime,the mummy murders,Leila Annastasia Scott Jason Scarbrough Jeff C...
1,Jake Johnson,Jake Johnson,Anna Kendrick,Natalie Morales,Comedy Thriller,self reliance,Jake Johnson Anna Kendrick Natalie Morales Jak...
2,Howard J. Ford,Ed Westwick,Andrew P. Stephen,Natalya Tsvetkova,Horror Thriller,darkgame,Ed Westwick Andrew P. Stephen Natalya Tsvetkov...
3,Bryce McGuire,Wyatt Russell,Kerry Condon,Amélie Hoeferle,Horror Mystery,night swim,Wyatt Russell Kerry Condon Amélie Hoeferle Bry...
4,Jeffrey Darling,Jacob Elordi,Zachary Quinto,Patrick J. Adams,Thriller Crime Drama,he went that way,Jacob Elordi Zachary Quinto Patrick J. Adams J...
...,...,...,...,...,...,...,...
490,James Mangold,Timothée Chalamet,Edward Norton,Elle Fanning,Drama Music,a complete unknown,Timothée Chalamet Edward Norton Elle Fanning J...
491,Rachel Morrison,Ryan Destiny,Brian Tyree Henry,unknown,Drama History,the fire inside,Ryan Destiny Brian Tyree Henry unknown Rachel ...
492,Halina Reijn,Nicole Kidman,Harris Dickinson,Sophie Wilde,Romance Thriller,babygirl,Nicole Kidman Harris Dickinson Sophie Wilde Ha...
493,"Michael Schwartz, Tyler Nilson (directors/scre...",Héctor Medina,Adria Arjona,Eros de la Puente,Drama,los frikis,Héctor Medina Adria Arjona Eros de la Puente M...


### Extracting features of 2025 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2025 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2025)

In [383]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2025"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2025 = tables[2]
df2_2025 = tables[3]
df3_2025 = tables[4]
df4_2025 = tables[5]

In [384]:
All_Scrabed_8 = pd.concat([df1_2025, df2_2025, df3_2025, df4_2025], ignore_index=True)
All_Scrabed_8

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,3,The Damned,Vertical / Ley Line Entertainment,Thordur Palsson (director); Jamie Hannigan (sc...,[3]
1,J A N U A R Y,10,Den of Thieves 2: Pantera,Lionsgate / Tucker Tooley Entertainment / G-BA...,Christian Gudegast (director/screenplay); Gera...,[4]
2,J A N U A R Y,10,Extremely Unique Dynamic,Strand Releasing,"Katherine Dudas, Ivan Leung, Harrison Xu (dire...",[5]
3,J A N U A R Y,10,Laws of Man,Saban Films,Phil Blattenberger (director/screenplay); Jaco...,[6]
4,J A N U A R Y,14,Man with No Past,Paramount Pictures,James Bamford (director); Steven Paul (screenp...,[7]
...,...,...,...,...,...,...
490,D E C E M B E R,25,Song Sung Blue,Focus Features / Davis Entertainment,Craig Brewer (director/screenplay); Hugh Jackm...,[470]
491,D E C E M B E R,25,Marty Supreme,A24,Josh Safdie (director/screenplay); Ronald Bron...,[471]
492,D E C E M B E R,25,The Testament of Ann Lee,Searchlight Pictures / Annapurna Pictures,Mona Fastvold (director/screenplay); Brady Cor...,[472]
493,D E C E M B E R,30,Continuance,BayView Entertainment / Rosewood Five,Tony Olmos (director/screenplay); Tony Gorodec...,[473]


In [385]:
All_Scrabed_8['genres'] = All_Scrabed_8['Title'].map(lambda x: get_genre(str(x)))

In [386]:
Scrapped_2025 = All_Scrabed_8[['Title','Cast and crew','genres']]
Scrapped_2025

,Title,Cast and crew,genres
0,The Damned,Thordur Palsson (director); Jamie Hannigan (sc...,Horror Drama Mystery
1,Den of Thieves 2: Pantera,Christian Gudegast (director/screenplay); Gera...,Action Crime Thriller
2,Extremely Unique Dynamic,"Katherine Dudas, Ivan Leung, Harrison Xu (dire...",Comedy Drama
3,Laws of Man,Phil Blattenberger (director/screenplay); Jaco...,Crime Drama Thriller
4,Man with No Past,James Bamford (director); Steven Paul (screenp...,Action Drama
...,...,...,...
490,Song Sung Blue,Craig Brewer (director/screenplay); Hugh Jackm...,Music Romance Drama
491,Marty Supreme,Josh Safdie (director/screenplay); Ronald Bron...,Drama Thriller
492,The Testament of Ann Lee,Mona Fastvold (director/screenplay); Brady Cor...,History Drama Music
493,Continuance,Tony Olmos (director/screenplay); Tony Gorodec...,Horror Comedy


In [387]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [388]:
Scrapped_2025['director_name'] = Scrapped_2025['Cast and crew'].map(lambda x: get_director(str(x)))

In [389]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [390]:
Scrapped_2025['actor_1_name'] = Scrapped_2025['Cast and crew'].map(lambda x: get_actor1(x))

In [391]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [392]:
Scrapped_2025['actor_2_name'] = Scrapped_2025['Cast and crew'].map(lambda x: get_actor2(x))

In [393]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [394]:
Scrapped_2025['actor_3_name'] = Scrapped_2025['Cast and crew'].map(lambda x: get_actor3(x))

In [395]:
Scrapped_2025 = Scrapped_2025.rename(columns={'Title':'movie_title'})

In [396]:
new_Scrapped_2025 = Scrapped_2025.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [397]:
new_Scrapped_2025['actor_2_name'] = new_Scrapped_2025['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2025['actor_3_name'] = new_Scrapped_2025['actor_3_name'].replace(np.nan, 'unknown')

In [398]:
new_Scrapped_2025['movie_title'] = new_Scrapped_2025['movie_title'].str.lower()

In [399]:
new_Scrapped_2025['comb'] = new_Scrapped_2025['actor_1_name'] + ' ' + new_Scrapped_2025['actor_2_name'] + ' '+ new_Scrapped_2025['actor_3_name'] + ' '+ new_Scrapped_2025['director_name'] +' ' + new_Scrapped_2025['genres']
new_Scrapped_2025

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Thordur Palsson,Odessa Young,Joe Cole,Siobhan Finneran,Horror Drama Mystery,the damned,Odessa Young Joe Cole Siobhan Finneran Thordur...
1,Christian Gudegast,Gerard Butler,O'Shea Jackson Jr.,Evin Ahmad,Action Crime Thriller,den of thieves 2: pantera,Gerard Butler O'Shea Jackson Jr. Evin Ahmad Ch...
2,"Katherine Dudas, Ivan Leung, Harrison Xu (dire...",Ivan Leung,Harrison Xu,Hudson Yang,Comedy Drama,extremely unique dynamic,Ivan Leung Harrison Xu Hudson Yang Katherine D...
3,Phil Blattenberger,Jacob Keohane,Jackson Rathbone,Richard Brake,Crime Drama Thriller,laws of man,Jacob Keohane Jackson Rathbone Richard Brake P...
4,James Bamford,Adam Woodward,Marton Csokas,Charlotte Weston,Action Drama,man with no past,Adam Woodward Marton Csokas Charlotte Weston J...
...,...,...,...,...,...,...,...
490,Craig Brewer,Hugh Jackman,Kate Hudson,Michael Imperioli,Music Romance Drama,song sung blue,Hugh Jackman Kate Hudson Michael Imperioli Cra...
491,Josh Safdie,Timothée Chalamet,Gwyneth Paltrow,Odessa A'zion,Drama Thriller,marty supreme,Timothée Chalamet Gwyneth Paltrow Odessa A'zio...
492,Mona Fastvold,Amanda Seyfried,Thomasin McKenzie,Lewis Pullman,History Drama Music,the testament of ann lee,Amanda Seyfried Thomasin McKenzie Lewis Pullma...
493,Tony Olmos,Tony Gorodeckas,Noor Razooky,Teresa Suarez Grosso,Horror Comedy,continuance,Tony Gorodeckas Noor Razooky Teresa Suarez Gro...


### Extracting features of 2026 movies from Wikipedia

The following is the data set used in this notebook Extracted from Wikipedia:

1. [The 2026 movies](https://en.wikipedia.org/wiki/List_of_American_films_of_2026)

In [400]:
link = "https://en.wikipedia.org/wiki/List_of_American_films_of_2026"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(link, headers=headers)

html = response.text

tables = pd.read_html(StringIO(html))

df1_2026 = tables[2]
df2_2026 = tables[3]
df3_2026 = tables[4]
df4_2026 = tables[5]

In [401]:
All_Scrabed_9 = pd.concat([df1_2026, df2_2026, df3_2026, df4_2026], ignore_index=True)
All_Scrabed_9

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,2,We Bury the Dead,Vertical,Zak Hilditch (director/screenplay); Daisy Ridl...,[3]
1,J A N U A R Y,2,The Dutchman,Rogue Pictures / Washington Square Films,Andre Gaines (director/screenplay); Qasim Basi...,[4]
2,J A N U A R Y,9,Primate,Paramount Pictures,Johannes Roberts (director/screenplay); Ernest...,[5]
3,J A N U A R Y,9,Greenland 2: Migration,Lionsgate / STXfilms / Thunder Road Films,Ric Roman Waugh (director); Mitchell LaFortune...,[6]
4,J A N U A R Y,9,Dead Man's Wire,Row K Entertainment,Gus Van Sant (director); Austin Kolodney (scre...,[7]
...,...,...,...,...,...,...
363,D E C E M B E R,25,Werwulf,Focus Features / Working Title Films / Maiden ...,Robert Eggers (director/screenplay); Sjón (scr...,[349]
364,D E C E M B E R,25,Mr. Irrelevant: The John Tuggle Story,Paramount Pictures / Skydance Sports,Jonathan Levine (director); Nick Santora (scre...,[350]
365,D E C E M B E R,25,A Place in Hell,Neon / Republic Pictures / MRC / T-Street Prod...,Chloe Domont (director/screenplay); Michelle W...,[351]
366,D E C E M B E R,25,Zero A. D.,Angel Studios,Alejandro Gómez Monteverde (director); Rod Bar...,[306]


In [402]:
All_Scrabed_9['genres'] = All_Scrabed_9['Title'].map(lambda x: get_genre(str(x)))

In [403]:
Scrapped_2026 = All_Scrabed_9[['Title','Cast and crew','genres']]
Scrapped_2026

,Title,Cast and crew,genres
0,We Bury the Dead,Zak Hilditch (director/screenplay); Daisy Ridl...,Horror Thriller
1,The Dutchman,Andre Gaines (director/screenplay); Qasim Basi...,Drama Thriller
2,Primate,Johannes Roberts (director/screenplay); Ernest...,Horror Thriller
3,Greenland 2: Migration,Ric Roman Waugh (director); Mitchell LaFortune...,Adventure Thriller Science Fiction
4,Dead Man's Wire,Gus Van Sant (director); Austin Kolodney (scre...,Crime Drama Thriller
...,...,...,...
363,Werwulf,Robert Eggers (director/screenplay); Sjón (scr...,Horror Mystery
364,Mr. Irrelevant: The John Tuggle Story,Jonathan Levine (director); Nick Santora (scre...,Drama
365,A Place in Hell,Chloe Domont (director/screenplay); Michelle W...,Thriller Drama
366,Zero A. D.,Alejandro Gómez Monteverde (director); Rod Bar...,Drama


In [404]:
def get_director(x):
    if " (director)" in x:
        return x.split(" (director)")[0]
    elif " (directors)" in x:
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0]

In [405]:
Scrapped_2026['director_name'] = Scrapped_2026['Cast and crew'].map(lambda x: get_director(str(x)))

In [406]:
def get_actor1(x):
    if isinstance(x, str):
        return (x.split("screenplay); ")[-1]).split(", ")[0]
    return None

In [407]:
Scrapped_2026['actor_1_name'] = Scrapped_2026['Cast and crew'].map(lambda x: get_actor1(x))

In [408]:
def get_actor2(x):
    if isinstance(x, str):
        actors = x.split("screenplay); ")[-1].split(", ")
        return actors[1] if len(actors) > 1 else np.nan
    return np.nan

In [409]:
Scrapped_2026['actor_2_name'] = Scrapped_2026['Cast and crew'].map(lambda x: get_actor2(x))

In [410]:
def get_actor3(x):
    if not isinstance(x, str):
        return np.nan

    actors = x.split("screenplay); ")[-1].split(", ")

    return actors[2] if len(actors) > 2 else np.nan

In [411]:
Scrapped_2026['actor_3_name'] = Scrapped_2026['Cast and crew'].map(lambda x: get_actor3(x))

In [412]:
Scrapped_2026 = Scrapped_2026.rename(columns={'Title':'movie_title'})

In [413]:
new_Scrapped_2026 = Scrapped_2026.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title']]

In [414]:
new_Scrapped_2026['actor_2_name'] = new_Scrapped_2026['actor_2_name'].replace(np.nan, 'unknown')
new_Scrapped_2026['actor_3_name'] = new_Scrapped_2026['actor_3_name'].replace(np.nan, 'unknown')

In [415]:
new_Scrapped_2026['movie_title'] = new_Scrapped_2026['movie_title'].str.lower()

In [416]:
new_Scrapped_2026['comb'] = new_Scrapped_2026['actor_1_name'] + ' ' + new_Scrapped_2026['actor_2_name'] + ' '+ new_Scrapped_2026['actor_3_name'] + ' '+ new_Scrapped_2026['director_name'] +' ' + new_Scrapped_2026['genres']
new_Scrapped_2026

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Zak Hilditch,Daisy Ridley,Mark Coles Smith,Brenton Thwaites,Horror Thriller,we bury the dead,Daisy Ridley Mark Coles Smith Brenton Thwaites...
1,Andre Gaines,André Holland,Kate Mara,Zazie Beetz,Drama Thriller,the dutchman,André Holland Kate Mara Zazie Beetz Andre Gain...
2,Johannes Roberts,Johnny Sequoyah,Jessica Alexander,Troy Kotsur,Horror Thriller,primate,Johnny Sequoyah Jessica Alexander Troy Kotsur ...
3,Ric Roman Waugh,Gerard Butler,Morena Baccarin,Roman Griffin Davis,Adventure Thriller Science Fiction,greenland 2: migration,Gerard Butler Morena Baccarin Roman Griffin Da...
4,Gus Van Sant,Bill Skarsgård,Dacre Montgomery,Cary Elwes,Crime Drama Thriller,dead man's wire,Bill Skarsgård Dacre Montgomery Cary Elwes Gus...
...,...,...,...,...,...,...,...
363,Robert Eggers,Aaron Taylor-Johnson,Willem Dafoe,Lily-Rose Depp,Horror Mystery,werwulf,Aaron Taylor-Johnson Willem Dafoe Lily-Rose De...
364,Jonathan Levine,David Corenswet,Isabel May,Michael Shannon,Drama,mr. irrelevant: the john tuggle story,David Corenswet Isabel May Michael Shannon Jon...
365,Chloe Domont,Michelle Williams,Daisy Edgar-Jones,Andrew Scott,Thriller Drama,a place in hell,Michelle Williams Daisy Edgar-Jones Andrew Sco...
366,Alejandro Gómez Monteverde,Deva Cassel,Sam Worthington,Ben Mendelsohn,Drama,zero a. d.,Deva Cassel Sam Worthington Ben Mendelsohn Ale...


In [417]:
all_movies = pd.concat([new_Scrapped_2018,new_Scrapped_2019,new_Scrapped_2020,new_Scrapped_2021,new_Scrapped_2022,new_Scrapped_2023,new_Scrapped_2024,new_Scrapped_2025,new_Scrapped_2026], ignore_index=True)
all_movies

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,insidious: the last key,Lin Shaye Angus Sampson Leigh Whannell Adam Ro...
1,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Drama Mystery,the strange ones,Lauren Wolkstein (director); Alex Pettyfer Jam...
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,the commuter,Liam Neeson Vera Farmiga Patrick Wilson Jaume ...
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,proud mary,Taraji P. Henson Jahi Di'Allo Winston Billy Br...
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,acts of violence,Bruce Willis Cole Hauser Shawn Ashmore Brett D...
...,...,...,...,...,...,...,...
3175,Robert Eggers,Aaron Taylor-Johnson,Willem Dafoe,Lily-Rose Depp,Horror Mystery,werwulf,Aaron Taylor-Johnson Willem Dafoe Lily-Rose De...
3176,Jonathan Levine,David Corenswet,Isabel May,Michael Shannon,Drama,mr. irrelevant: the john tuggle story,David Corenswet Isabel May Michael Shannon Jon...
3177,Chloe Domont,Michelle Williams,Daisy Edgar-Jones,Andrew Scott,Thriller Drama,a place in hell,Michelle Williams Daisy Edgar-Jones Andrew Sco...
3178,Alejandro Gómez Monteverde,Deva Cassel,Sam Worthington,Ben Mendelsohn,Drama,zero a. d.,Deva Cassel Sam Worthington Ben Mendelsohn Ale...


In [418]:
all_movies.to_csv(r'D:\Projects\Ai\Recomendation system\Datasets\movie_scraped.csv',index=False)

In [419]:
movie_scraped = pd.read_csv(r'D:\Projects\Ai\Recomendation system\Datasets\movie_scraped.csv')
movie_scraped

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,insidious: the last key,Lin Shaye Angus Sampson Leigh Whannell Adam Ro...
1,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Drama Mystery,the strange ones,Lauren Wolkstein (director); Alex Pettyfer Jam...
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,the commuter,Liam Neeson Vera Farmiga Patrick Wilson Jaume ...
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,proud mary,Taraji P. Henson Jahi Di'Allo Winston Billy Br...
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,acts of violence,Bruce Willis Cole Hauser Shawn Ashmore Brett D...
...,...,...,...,...,...,...,...
3175,Robert Eggers,Aaron Taylor-Johnson,Willem Dafoe,Lily-Rose Depp,Horror Mystery,werwulf,Aaron Taylor-Johnson Willem Dafoe Lily-Rose De...
3176,Jonathan Levine,David Corenswet,Isabel May,Michael Shannon,Drama,mr. irrelevant: the john tuggle story,David Corenswet Isabel May Michael Shannon Jon...
3177,Chloe Domont,Michelle Williams,Daisy Edgar-Jones,Andrew Scott,Thriller Drama,a place in hell,Michelle Williams Daisy Edgar-Jones Andrew Sco...
3178,Alejandro Gómez Monteverde,Deva Cassel,Sam Worthington,Ben Mendelsohn,Drama,zero a. d.,Deva Cassel Sam Worthington Ben Mendelsohn Ale...


In [420]:
movie_reprocessed_Data_2 = pd.read_csv(r'D:\Projects\Ai\Recomendation system\Datasets\movie_reprocessed_2.csv')
movie_reprocessed_Data_2

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
5359,Jim Strouse,Jessica Williams,Chris O'Dowd,Keith Stanfield,Romance Comedy,the incredible jessica james,Jessica Williams Chris O'Dowd Keith Stanfield ...
5360,Farhad Mann,Adelaide Kane,Benjamin Hollingsworth,Jean Louisa Kelly,Romance,can't buy my love,Adelaide Kane Benjamin Hollingsworth Jean Loui...
5361,Hannaleena Hauru,Inka Haapamäki,Rosa Honkonen,Tiitus Rantala,Romance Comedy,thick lashes of lauri mäntyvaara,Inka Haapamäki Rosa Honkonen Tiitus Rantala Ha...
5362,Jonathan A. Rosenbaum,Lou Diamond Phillips,Wallace Shawn,Gina Holden,Crime Comedy Action Family,cop and a half: new recruit,Lou Diamond Phillips Wallace Shawn Gina Holden...


In [421]:
All_Data_movies = pd.concat([movie_reprocessed_Data_2, movie_scraped], ignore_index=True)
All_Data_movies

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
8539,Robert Eggers,Aaron Taylor-Johnson,Willem Dafoe,Lily-Rose Depp,Horror Mystery,werwulf,Aaron Taylor-Johnson Willem Dafoe Lily-Rose De...
8540,Jonathan Levine,David Corenswet,Isabel May,Michael Shannon,Drama,mr. irrelevant: the john tuggle story,David Corenswet Isabel May Michael Shannon Jon...
8541,Chloe Domont,Michelle Williams,Daisy Edgar-Jones,Andrew Scott,Thriller Drama,a place in hell,Michelle Williams Daisy Edgar-Jones Andrew Sco...
8542,Alejandro Gómez Monteverde,Deva Cassel,Sam Worthington,Ben Mendelsohn,Drama,zero a. d.,Deva Cassel Sam Worthington Ben Mendelsohn Ale...


In [423]:
All_Data_movies.drop_duplicates(subset ="movie_title", keep = 'last', inplace = True)
All_Data_movies

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,comb
0,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,Action Adventure Fantasy Sci-Fi,avatar,CCH Pounder Joel David Moore Wes Studi James C...
1,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,Action Adventure Fantasy,pirates of the caribbean: at world's end,Johnny Depp Orlando Bloom Jack Davenport Gore ...
2,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,Action Adventure Thriller,spectre,Christoph Waltz Rory Kinnear Stephanie Sigman ...
3,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,Action Thriller,the dark knight rises,Tom Hardy Christian Bale Joseph Gordon-Levitt ...
4,Doug Walker,Doug Walker,Rob Walker,unknown,Documentary,star wars: episode vii - the force awakens ...,Doug Walker Rob Walker unknown Doug Walker Doc...
...,...,...,...,...,...,...,...
8539,Robert Eggers,Aaron Taylor-Johnson,Willem Dafoe,Lily-Rose Depp,Horror Mystery,werwulf,Aaron Taylor-Johnson Willem Dafoe Lily-Rose De...
8540,Jonathan Levine,David Corenswet,Isabel May,Michael Shannon,Drama,mr. irrelevant: the john tuggle story,David Corenswet Isabel May Michael Shannon Jon...
8541,Chloe Domont,Michelle Williams,Daisy Edgar-Jones,Andrew Scott,Thriller Drama,a place in hell,Michelle Williams Daisy Edgar-Jones Andrew Sco...
8542,Alejandro Gómez Monteverde,Deva Cassel,Sam Worthington,Ben Mendelsohn,Drama,zero a. d.,Deva Cassel Sam Worthington Ben Mendelsohn Ale...


In [424]:
All_Data_movies.to_csv(r'D:\Projects\Ai\Recomendation system\Datasets\All_Data_movies.csv',index=False)